# Farabi — TF-IDF & Classical Machine Learning

**Assigned contribution:** TF-IDF representation and manually tuned classical classifiers:
Random Forest, Logistic Regression and Multinomial Naive Bayes.

> Shared setup/preprocessing cells are included so the notebook can run independently.
> Farabi's primary contribution begins at **Part A — TF-IDF + Classical Machine Learning**.


2. Dataset Upload

In [2]:
from google.colab import files

uploaded = files.upload()

!pip -q install gensim datasets transformers accelerate
!unzip -o "/content/archive (4).zip" -d "/content/"

print("Environment setup completed and dataset archive extracted.")

Saving archive (4).zip to archive (4).zip
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 57.4 MB/s eta 0:00:00
Archive:  /content/archive (4).zip
  inflating: /content/cyberbullying_tweets.csv  
Environment setup completed and dataset archive extracted.


## 3. Imports and Reproducibility



In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nltk
import tensorflow as tf
import torch

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.probability import FreqDist
from wordcloud import WordCloud

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from gensim.models import Word2Vec

from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, GRU, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

physical_gpus = tf.config.list_physical_devices('GPU')
for gpu in physical_gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

print("Imports completed.")
print("TensorFlow GPU devices:", physical_gpus)
print("PyTorch CUDA available:", torch.cuda.is_available())

Imports completed.
TensorFlow GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
PyTorch CUDA available: True


## 5. Dataset Loading and Initial Inspection

The supplied CSV contains two columns: tweet text and the cyberbullying class label.

In [4]:
DATA_PATH = "/content/cyberbullying_tweets.csv"
df = pd.read_csv(DATA_PATH)

print("Raw dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Number of classes:", df["cyberbullying_type"].nunique())
df.head()

Raw dataset shape: (47692, 2)
Columns: ['tweet_text', 'cyberbullying_type']
Number of classes: 6


,tweet_text,cyberbullying_type
0,"In other words #katandandre, your food was cra...",not_cyberbullying
1,Why is #aussietv so white? #MKR #theblock #ImA...,not_cyberbullying
2,@XochitlSuckkks a classy whore? Or more red ve...,not_cyberbullying
3,"@Jason_Gio meh. :P thanks for the heads up, b...",not_cyberbullying
4,@RudhoeEnglish This is an ISIS account pretend...,not_cyberbullying


## 6. Data Quality Analysis

The project requires missing-value and duplicate analysis. In addition to exact duplicate rows, this dataset contains repeated tweet texts. Some repeated texts are associated with more than one label, which would create contradictory supervision and possible leakage if the same text appears in different splits.

The cleanup policy is:
1. Remove rows with missing text or missing labels.
2. Identify tweet texts assigned to multiple different labels and remove all rows for those contradictory texts.
3. Remove remaining duplicate tweet texts with the same label.
4. Perform the train/validation/test split only after this cleanup.

In [5]:
missing_summary = df.isna().sum()
exact_duplicate_rows = df.duplicated().sum()
unique_texts = df["tweet_text"].nunique()
repeated_text_extra_rows = len(df) - unique_texts

label_counts_per_text = df.groupby("tweet_text")["cyberbullying_type"].nunique()
conflicting_texts = label_counts_per_text[label_counts_per_text > 1].index
conflicting_rows = df[df["tweet_text"].isin(conflicting_texts)].shape[0]

quality_summary = pd.DataFrame({
    "Measure": [
        "Raw rows",
        "Missing tweet_text",
        "Missing cyberbullying_type",
        "Exact duplicate rows",
        "Extra rows caused by repeated tweet text",
        "Tweet texts with conflicting labels",
        "Rows involved in conflicting labels"
    ],
    "Value": [
        len(df),
        missing_summary["tweet_text"],
        missing_summary["cyberbullying_type"],
        exact_duplicate_rows,
        repeated_text_extra_rows,
        len(conflicting_texts),
        conflicting_rows
    ]
})
quality_summary

,Measure,Value
0,Raw rows,47692
1,Missing tweet_text,0
2,Missing cyberbullying_type,0
3,Exact duplicate rows,36
4,Extra rows caused by repeated tweet text,1675
5,Tweet texts with conflicting labels,1639
6,Rows involved in conflicting labels,3278


In [6]:
df = df.dropna(subset=["tweet_text", "cyberbullying_type"]).copy()

label_counts_per_text = df.groupby("tweet_text")["cyberbullying_type"].nunique()
conflicting_texts = label_counts_per_text[label_counts_per_text > 1].index

df = df[~df["tweet_text"].isin(conflicting_texts)].copy()
df = df.drop_duplicates(subset=["tweet_text"]).reset_index(drop=True)

print("Dataset shape after quality cleanup:", df.shape)
print("Remaining duplicate tweet texts:", df["tweet_text"].duplicated().sum())
print("Remaining missing values:")
print(df.isna().sum())

Dataset shape after quality cleanup: (44378, 2)
Remaining duplicate tweet texts: 0
Remaining missing values:
tweet_text            0
cyberbullying_type    0
dtype: int64


## 8. Label Encoding and Stratified Train/Validation/Test Split

The cleaned dataset is split **70% / 15% / 15%**. Stratification preserves the six-class distribution in every partition and `random_state=42` makes the split reproducible.

In [7]:
label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["cyberbullying_type"])

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=df["label"]
)

validation_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=temp_df["label"]
)

train_df = train_df.reset_index(drop=True)
validation_df = validation_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Class mapping:", dict(enumerate(label_encoder.classes_)))
print("Train shape:", train_df.shape)
print("Validation shape:", validation_df.shape)
print("Test shape:", test_df.shape)

Class mapping: {0: 'age', 1: 'ethnicity', 2: 'gender', 3: 'not_cyberbullying', 4: 'other_cyberbullying', 5: 'religion'}
Train shape: (31064, 3)
Validation shape: (6657, 3)
Test shape: (6657, 3)


## 9. Preprocessing Strategies

Four preprocessing strategies are built from Lab 1 operations. Their effect is tested with the same TF-IDF + Logistic Regression validation baseline so that preprocessing is selected empirically instead of by assumption.

- **Lowercase only:** retains all symbols, hashtags and punctuation after lowercasing.
- **Alphabetic tokens:** lowercases, tokenizes and retains alphabetic tokens only.
- **Stopword + stemming:** adds English stopword removal and Porter stemming.
- **Stopword + lemmatization:** adds English stopword removal and WordNet lemmatization.

In [8]:
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
print("NLTK tokenization resources are ready.")


NLTK tokenization resources are ready.


In [9]:
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()


def preprocess_lowercase(text):
    return str(text).lower()


def preprocess_alpha(text):
    words = word_tokenize(str(text).lower())
    filtered_words = []
    for word in words:
        if word.isalpha():
            filtered_words.append(word)
    return " ".join(filtered_words)


def preprocess_stem(text):
    words = word_tokenize(str(text).lower())
    filtered_words = []
    for word in words:
        if word.isalpha() and word not in stop_words:
            filtered_words.append(stemmer.stem(word))
    return " ".join(filtered_words)


def preprocess_lemma(text):
    words = word_tokenize(str(text).lower())
    filtered_words = []
    for word in words:
        if word.isalpha() and word not in stop_words:
            filtered_words.append(lemmatizer.lemmatize(word, pos="v"))
    return " ".join(filtered_words)


preprocessing_functions = {
    "Lowercase only": preprocess_lowercase,
    "Alphabetic tokens": preprocess_alpha,
    "Stopword + stemming": preprocess_stem,
    "Stopword + lemmatization": preprocess_lemma
}

print("Preprocessing functions defined:", list(preprocessing_functions.keys()))

Preprocessing functions defined: ['Lowercase only', 'Alphabetic tokens', 'Stopword + stemming', 'Stopword + lemmatization']


### 9.1 Validation-Based Preprocessing Selection

Each preprocessing strategy is fitted only on the training split through a TF-IDF vectorizer, then evaluated on the validation split using Logistic Regression. The strategy with the highest validation **Macro-F1** is selected for the classical and recurrent-model experiments.

In [10]:
preprocessing_results = []

for name, function in preprocessing_functions.items():
    train_text = train_df["tweet_text"].apply(function)
    validation_text = validation_df["tweet_text"].apply(function)

    preprocessing_tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words="english")
    X_train_preprocessing = preprocessing_tfidf.fit_transform(train_text)
    X_validation_preprocessing = preprocessing_tfidf.transform(validation_text)

    preprocessing_model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
    preprocessing_model.fit(X_train_preprocessing, train_df["label"])
    validation_prediction = preprocessing_model.predict(X_validation_preprocessing)

    validation_accuracy = accuracy_score(validation_df["label"], validation_prediction)
    validation_f1 = f1_score(validation_df["label"], validation_prediction, average="macro")

    preprocessing_results.append({
        "Preprocessing": name,
        "Validation Accuracy": validation_accuracy,
        "Validation Macro F1": validation_f1
    })

preprocessing_results_df = pd.DataFrame(preprocessing_results).sort_values(
    "Validation Macro F1", ascending=False
).reset_index(drop=True)

preprocessing_results_df

,Preprocessing,Validation Accuracy,Validation Macro F1
0,Lowercase only,0.868259,0.855946
1,Stopword + lemmatization,0.864203,0.851797
2,Stopword + stemming,0.863302,0.850771
3,Alphabetic tokens,0.861499,0.849439


In [11]:
best_preprocessing_name = preprocessing_results_df.loc[0, "Preprocessing"]
best_preprocessing_function = preprocessing_functions[best_preprocessing_name]

train_df["clean_text"] = train_df["tweet_text"].apply(best_preprocessing_function)
validation_df["clean_text"] = validation_df["tweet_text"].apply(best_preprocessing_function)
test_df["clean_text"] = test_df["tweet_text"].apply(best_preprocessing_function)

print("Selected preprocessing strategy:", best_preprocessing_name)
print("Example cleaned tweet:")
print(train_df.loc[0, "clean_text"])

Selected preprocessing strategy: Lowercase only
Example cleaned tweet:
man this nigger dumb as fuck rt "@lumkile1st: lmao. fake weed? "@mynameztom: the cops are giving people fake weed then arresting them""


## 10. Shared Experiment Tracking and Evaluation Functions

Every required model is tuned with at least three explicit configurations. Validation Accuracy and validation Macro-F1 are recorded for every run. The final test evaluation records Accuracy, Macro-F1, the full classification report and a confusion matrix.

In [12]:
tuning_records = []
final_results = []
confusion_matrices = {}
best_hyperparameters = {}


def record_tuning(model_name, representation, config_id, parameters, validation_accuracy, validation_f1):
    tuning_records.append({
        "Model": model_name,
        "Representation": representation,
        "Config": config_id,
        "Parameters": str(parameters),
        "Validation Accuracy": validation_accuracy,
        "Validation Macro F1": validation_f1
    })


def evaluate_predictions(model_name, representation, y_true, y_pred):
    test_accuracy = accuracy_score(y_true, y_pred)
    test_f1 = f1_score(y_true, y_pred, average="macro")
    test_confusion_matrix = confusion_matrix(y_true, y_pred)

    print("=" * 90)
    print(model_name)
    print("Representation:", representation)
    print("Test Accuracy:", round(test_accuracy, 4))
    print("Test Macro F1:", round(test_f1, 4))
    print("\nFull Classification Report:\n")
    print(classification_report(y_true, y_pred, target_names=label_encoder.classes_, zero_division=0))
    print("Confusion Matrix:\n", test_confusion_matrix)

    final_results.append({
        "Model": model_name,
        "Representation": representation,
        "Accuracy": test_accuracy,
        "Macro F1": test_f1
    })
    confusion_matrices[model_name] = test_confusion_matrix

    return test_accuracy, test_f1


print("Experiment tracking and evaluation functions are ready.")

Experiment tracking and evaluation functions are ready.


# Part A — TF-IDF + Classical Machine Learning

## 11. TF-IDF Representation

TF-IDF is fitted **only on the training split**, then the same fitted vectorizer transforms validation and test data. This prevents vocabulary and document-frequency leakage from the held-out sets.

In [13]:
tfidf_vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    stop_words="english"
)

X_train_tfidf = tfidf_vectorizer.fit_transform(train_df["clean_text"])
X_validation_tfidf = tfidf_vectorizer.transform(validation_df["clean_text"])
X_test_tfidf = tfidf_vectorizer.transform(test_df["clean_text"])

y_train = train_df["label"].values
y_validation = validation_df["label"].values
y_test = test_df["label"].values

print("TF-IDF train shape:", X_train_tfidf.shape)
print("TF-IDF validation shape:", X_validation_tfidf.shape)
print("TF-IDF test shape:", X_test_tfidf.shape)

TF-IDF train shape: (31064, 10000)
TF-IDF validation shape: (6657, 10000)
TF-IDF test shape: (6657, 10000)


## 12. Random Forest — Manual Hyperparameter Tuning

Three configurations vary the number of trees and maximum tree depth. The configuration with the highest validation Macro-F1 is retained for the test set.

In [14]:
rf_configs = [
    {"n_estimators": 100, "max_depth": None},
    {"n_estimators": 200, "max_depth": 50},
    {"n_estimators": 300, "max_depth": 100}
]

best_rf_model = None
best_rf_config = None
best_rf_validation_f1 = -1

for config_id, config in enumerate(rf_configs, start=1):
    model = RandomForestClassifier(
        n_estimators=config["n_estimators"],
        max_depth=config["max_depth"],
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    model.fit(X_train_tfidf, y_train)
    validation_prediction = model.predict(X_validation_tfidf)

    validation_accuracy = accuracy_score(y_validation, validation_prediction)
    validation_f1 = f1_score(y_validation, validation_prediction, average="macro")

    record_tuning("Random Forest", "TF-IDF", config_id, config, validation_accuracy, validation_f1)
    print("Config", config_id, config, "Validation Macro F1 =", round(validation_f1, 4))

    if validation_f1 > best_rf_validation_f1:
        best_rf_validation_f1 = validation_f1
        best_rf_model = model
        best_rf_config = config

best_hyperparameters["Random Forest"] = best_rf_config
print("Best Random Forest config:", best_rf_config)

Config 1 {'n_estimators': 100, 'max_depth': None} Validation Macro F1 = 0.865
Config 2 {'n_estimators': 200, 'max_depth': 50} Validation Macro F1 = 0.828
Config 3 {'n_estimators': 300, 'max_depth': 100} Validation Macro F1 = 0.8465
Best Random Forest config: {'n_estimators': 100, 'max_depth': None}


In [15]:
rf_test_prediction = best_rf_model.predict(X_test_tfidf)
evaluate_predictions("Random Forest", "TF-IDF", y_test, rf_test_prediction)

Random Forest
Representation: TF-IDF
Test Accuracy: 0.8777
Test Macro F1: 0.8654

Full Classification Report:

                     precision    recall  f1-score   support

                age       0.98      0.98      0.98      1199
          ethnicity       0.98      0.98      0.98      1193
             gender       0.92      0.87      0.89      1166
  not_cyberbullying       0.70      0.63      0.66       964
other_cyberbullying       0.67      0.78      0.72       937
           religion       0.96      0.96      0.96      1198

           accuracy                           0.88      6657
          macro avg       0.87      0.87      0.87      6657
       weighted avg       0.88      0.88      0.88      6657

Confusion Matrix:
 [[1176    1    0   16    5    1]
 [   1 1164    3    9   14    2]
 [   1    2 1015   72   76    0]
 [  19    7   39  607  244   48]
 [   8    9   45  143  729    3]
 [   0    0    5   23   18 1152]]


(0.8777226979119723, 0.8653859636179577)

## 13. Logistic Regression — Manual Hyperparameter Tuning

The regularization parameter `C` is varied across three configurations while keeping the rest of the Lab 1/2 workflow fixed.

In [16]:
lr_configs = [
    {"C": 0.5},
    {"C": 1.0},
    {"C": 2.0}
]

best_lr_model = None
best_lr_config = None
best_lr_validation_f1 = -1

for config_id, config in enumerate(lr_configs, start=1):
    model = LogisticRegression(
        C=config["C"],
        max_iter=1000,
        random_state=RANDOM_STATE
    )
    model.fit(X_train_tfidf, y_train)
    validation_prediction = model.predict(X_validation_tfidf)

    validation_accuracy = accuracy_score(y_validation, validation_prediction)
    validation_f1 = f1_score(y_validation, validation_prediction, average="macro")

    record_tuning("Logistic Regression", "TF-IDF", config_id, config, validation_accuracy, validation_f1)
    print("Config", config_id, config, "Validation Macro F1 =", round(validation_f1, 4))

    if validation_f1 > best_lr_validation_f1:
        best_lr_validation_f1 = validation_f1
        best_lr_model = model
        best_lr_config = config

best_hyperparameters["Logistic Regression"] = best_lr_config
print("Best Logistic Regression config:", best_lr_config)

Config 1 {'C': 0.5} Validation Macro F1 = 0.8556
Config 2 {'C': 1.0} Validation Macro F1 = 0.8564
Config 3 {'C': 2.0} Validation Macro F1 = 0.8569
Best Logistic Regression config: {'C': 2.0}


In [17]:
lr_test_prediction = best_lr_model.predict(X_test_tfidf)
evaluate_predictions("Logistic Regression", "TF-IDF", y_test, lr_test_prediction)

Logistic Regression
Representation: TF-IDF
Test Accuracy: 0.8791
Test Macro F1: 0.8683

Full Classification Report:

                     precision    recall  f1-score   support

                age       0.96      0.98      0.97      1199
          ethnicity       0.98      0.97      0.98      1193
             gender       0.93      0.87      0.90      1166
  not_cyberbullying       0.67      0.68      0.68       964
other_cyberbullying       0.72      0.76      0.74       937
           religion       0.95      0.95      0.95      1198

           accuracy                           0.88      6657
          macro avg       0.87      0.87      0.87      6657
       weighted avg       0.88      0.88      0.88      6657

Confusion Matrix:
 [[1170    2    1   19    6    1]
 [   2 1161    2   12   14    2]
 [   5    7 1019   69   63    3]
 [  33    7   42  656  183   43]
 [   8   10   32  168  712    7]
 [   0    1    4   48   11 1134]]


(0.879074658254469, 0.8682559774589627)

## 14. Multinomial Naive Bayes — Manual Hyperparameter Tuning

The Laplace/additive-smoothing parameter `alpha` is varied across three configurations.

In [18]:
nb_configs = [
    {"alpha": 0.5},
    {"alpha": 1.0},
    {"alpha": 1.5}
]

best_nb_model = None
best_nb_config = None
best_nb_validation_f1 = -1

for config_id, config in enumerate(nb_configs, start=1):
    model = MultinomialNB(alpha=config["alpha"])
    model.fit(X_train_tfidf, y_train)
    validation_prediction = model.predict(X_validation_tfidf)

    validation_accuracy = accuracy_score(y_validation, validation_prediction)
    validation_f1 = f1_score(y_validation, validation_prediction, average="macro")

    record_tuning("Naive Bayes", "TF-IDF", config_id, config, validation_accuracy, validation_f1)
    print("Config", config_id, config, "Validation Macro F1 =", round(validation_f1, 4))

    if validation_f1 > best_nb_validation_f1:
        best_nb_validation_f1 = validation_f1
        best_nb_model = model
        best_nb_config = config

best_hyperparameters["Naive Bayes"] = best_nb_config
print("Best Naive Bayes config:", best_nb_config)

Config 1 {'alpha': 0.5} Validation Macro F1 = 0.7894
Config 2 {'alpha': 1.0} Validation Macro F1 = 0.7867
Config 3 {'alpha': 1.5} Validation Macro F1 = 0.7827
Best Naive Bayes config: {'alpha': 0.5}


In [19]:
nb_test_prediction = best_nb_model.predict(X_test_tfidf)
evaluate_predictions("Naive Bayes", "TF-IDF", y_test, nb_test_prediction)

Naive Bayes
Representation: TF-IDF
Test Accuracy: 0.8152
Test Macro F1: 0.793

Full Classification Report:

                     precision    recall  f1-score   support

                age       0.80      0.98      0.88      1199
          ethnicity       0.90      0.91      0.91      1193
             gender       0.87      0.83      0.85      1166
  not_cyberbullying       0.71      0.50      0.59       964
other_cyberbullying       0.72      0.58      0.64       937
           religion       0.82      0.97      0.89      1198

           accuracy                           0.82      6657
          macro avg       0.80      0.80      0.79      6657
       weighted avg       0.81      0.82      0.81      6657

Confusion Matrix:
 [[1175    3    2    8    9    2]
 [  24 1091   10    7   27   34]
 [  17   32  969   72   52   24]
 [ 126   38   57  484  118  141]
 [ 129   51   65   91  548   53]
 [   5    2    5   16   10 1160]]


(0.815232086525462, 0.7930336189092628)